['actions', 'pot_start', 'pot_states1', 'pot_states2', 'robot0_eye_in_hand_image', 'robot1_eye_in_hand_image']
(649, 64, 64, 3)


: 

: 

: 

In [1]:
!pip install numpy matplotlib torch diffusers scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 136.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 84.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 34.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [diffusers]19 [diffusers]e-hub]er]


In [2]:
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 86.7 MB/s  0:00:00


In [9]:
!pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.5 MB/s  0:00:00


In [10]:
# Parses the rollout data from demo.py and saves them as numpy arrays of states, actions, and pot handle positions (the conditional vector data)

import numpy as np
import csv
import matplotlib.pyplot as plt
import pickle as pkl
import torch
from scipy.spatial.transform import Rotation as R
from diffusers import AutoencoderKL
import os
import sys

# Add parent dir (lift/) and encoder_scripts/ to path so imports resolve
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath(os.path.join("..", "encoder_scripts")))

from transform_utils import quat_to_rot6d, rotvec_to_rot6d, rot6d_to_quat
from net import TimMResNet18Encoder


In [ ]:
# Initialize ViT encoder for image processing
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# vit_encoder = VisionTransformerEncoder(latent_dim=128)
# vit_encoder.to(device).eval()

: 

: 

In [16]:
# Example usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = TimMResNet18Encoder(pretrained=True, latent_dim=128).to(device)
encoder.eval()

TimMResNet18Encoder(
  (backbone): FeatureListNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=

In [43]:
# with open("rollouts_pot/rollout_seed0_mode2.pkl", "rb") as f:
#     rollout = pkl.load(f)
#     obs = rollout["observations"]
#     actions = np.array(rollout["actions"])
#     robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
#     robot0_eef_quat = np.array([o["robot0_eef_quat"] for o in obs])
#     robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
#     robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
#     robot1_eef_quat = np.array([o["robot1_eef_quat"] for o in obs])
#     robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

#     repeats_needed = 250 - actions.shape[0]

#     repeated_last = np.tile(actions[-1], (repeats_needed, 1))
#     actions = np.vstack([actions, repeated_last])

#     repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
#     robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
#     state = robot0_eef_pos

#     repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
#     robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
#     robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
#     state = np.hstack([state, robot0_eef_rotvec])


#     repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
#     robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
#     robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
#     state = np.hstack([state, robot0_gripper_pos])

#     repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
#     robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
#     state = np.hstack([state, robot1_eef_pos])

#     repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
#     robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
#     robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
#     state = np.hstack([state, robot1_eef_rotvec])

#     repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
#     robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
#     robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
#     state = np.hstack([state, robot1_gripper_pos])

# print(np.shape(state))
# print(np.shape(actions))

In [ ]:
# Use PRE-TRAINED ResNet for image encoding (NO training needed!)
# import torchvision.models as models
# import torch.nn as nn

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Load pre-trained ResNet18
# resnet = models.resnet18(pretrained=True)
# encoder_backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC layer
# projection = nn.Linear(512, 128)  # Project to 128 dims

# # Combined encoder
# class vit_encoder_class(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.backbone = encoder_backbone
#         self.proj = projection

#     def forward(self, x):
#         # Normalize with ImageNet stats
#         mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device)
#         std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device)
#         x = (x - mean) / std
#         features = self.backbone(x).view(x.size(0), -1)
#         return self.proj(features)

# vit_encoder = vit_encoder_class().to(device).eval()
# print("✅ Using pre-trained ResNet18 encoder!")

['/content/drive/MyDrive/VAE_models_ICON/best_vae.pth']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

boss


AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

In [ ]:
expert_states_list = []
expert_actions_list = []
pot_start_list = []
pot_states_list1 = []
pot_states_list2 = []
image_latents_list0 = []
image_latents_list1 = []

import glob
robust_dir = "../rollouts/robust"
robust_files = sorted(glob.glob(os.path.join(robust_dir, "*.pkl")))
print(f"Found {len(robust_files)} rollout files in {robust_dir}")

for file_idx, filepath in enumerate(robust_files):
    with open(filepath, "rb") as f:
            rollout = pkl.load(f)
            obs = rollout["observations"]
            actions = np.array(rollout["actions"])
            print(np.shape(actions))
            print(f"[{file_idx+1}/{len(robust_files)}] {os.path.basename(filepath)}")
            pot1 = np.array(rollout["pot_states1"])
            pot2 = np.array(rollout["pot_states2"])
            pot = np.array(rollout["pot_start"])

            pot_start_list.append(np.concatenate((pot[0], pot[1])))

            T_target = 700

            if "camera_obs0" in rollout and "camera_obs1" in rollout:
                camera0_obs = np.array(rollout["camera_obs0"])  # (T, H, W, C)
                camera1_obs = np.array(rollout["camera_obs1"])

                # Pad or truncate images to T_target
                if camera0_obs.shape[0] > T_target:
                    camera0_obs = camera0_obs[:T_target]
                    camera1_obs = camera1_obs[:T_target]
                elif camera0_obs.shape[0] < T_target:
                    repeats_needed = T_target - camera0_obs.shape[0]
                    camera0_obs = np.vstack([camera0_obs, np.repeat(camera0_obs[-1][None], repeats_needed, axis=0)])
                    camera1_obs = np.vstack([camera1_obs, np.repeat(camera1_obs[-1][None], repeats_needed, axis=0)])

                camera0_latents = []
                camera1_latents = []

                for frame_idx in range(T_target):
                    img0 = torch.from_numpy(camera0_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
                    img1 = torch.from_numpy(camera1_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

                    with torch.no_grad():
                        latents0 = encoder(img0)
                        latents1 = encoder(img1)

                    camera0_latents.append(latents0.cpu().numpy().squeeze())
                    camera1_latents.append(latents1.cpu().numpy().squeeze())

                camera0_latents = np.array(camera0_latents)
                camera1_latents = np.array(camera1_latents)
            else:
                print("Womp womp :(")

            robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
            robot0_eef_quat = np.array([o["robot0_eef_quat_site"] for o in obs])
            robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
            robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
            robot1_eef_quat = np.array([o["robot1_eef_quat_site"] for o in obs])
            robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

            T = actions.shape[0]
            if T > 700:
                # Truncate all arrays to 700
                actions = actions[:700]
                robot0_eef_pos = robot0_eef_pos[:700]
                robot0_eef_quat = robot0_eef_quat[:700]
                robot0_gripper_pos = robot0_gripper_pos[:700]
                robot1_eef_pos = robot1_eef_pos[:700]
                robot1_eef_quat = robot1_eef_quat[:700]
                robot1_gripper_pos = robot1_gripper_pos[:700]
                pot1 = pot1[:700]
                pot2 = pot2[:700]
                repeats_needed = 0
            else:
                repeats_needed = 700 - T

            if repeats_needed > 0:
                repeated_last = np.tile(actions[-1], (repeats_needed, 1))
                actions = np.vstack([actions, repeated_last])

                repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
                robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])

                repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
                robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])

                repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed,))
                robot0_gripper_pos = np.concatenate([robot0_gripper_pos, repeated_last])

                repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
                robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])

                repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
                robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])

                repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed,))
                robot1_gripper_pos = np.concatenate([robot1_gripper_pos, repeated_last])

                repeated_last = np.tile(pot1[-1], (repeats_needed, 1))
                pot1 = np.vstack([pot1, repeated_last])
                repeated_last = np.tile(pot2[-1], (repeats_needed, 1))
                pot2 = np.vstack([pot2, repeated_last])

            # Build state vector
            state = robot0_eef_pos
            robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
            state = np.hstack([state, robot0_eef_rotvec])
            robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
            state = np.hstack([state, robot0_gripper_pos])
            state = np.hstack([state, robot1_eef_pos])
            robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
            state = np.hstack([state, robot1_eef_rotvec])
            robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
            state = np.hstack([state, robot1_gripper_pos])

            pot_states_list1.append(pot1)
            pot_states_list2.append(pot2)

            # --- ENFORCE FINAL SHAPE FOR LATENTS --- #
            T_target = 700

            # Camera 0
            if camera0_latents.shape[0] > T_target:
                camera0_latents = camera0_latents[:T_target]
            elif camera0_latents.shape[0] < T_target:
                repeats_needed = T_target - camera0_latents.shape[0]
                last_frame = camera0_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera0_latents = np.concatenate([camera0_latents, padding], axis=0)

            if camera1_latents.shape[0] > T_target:
                camera1_latents = camera1_latents[:T_target]
            elif camera1_latents.shape[0] < T_target:
                repeats_needed = T_target - camera1_latents.shape[0]
                last_frame = camera1_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera1_latents = np.concatenate([camera1_latents, padding], axis=0)

            print("ight blud")
            print(state.shape)
            print(camera0_latents.shape)
            print(actions.shape)
                

            image_latents_list0.append(camera0_latents)
            image_latents_list1.append(camera1_latents)


            expert_states_list.append(state)
            expert_actions_list.append(actions)



expert_states_rotvec = np.stack(expert_states_list, axis=0)
expert_actions_rotvec = np.stack(expert_actions_list, axis=0)
pot_states_rotvec1 = np.stack(pot_states_list1, axis=0)
pot_states_rotvec2 = np.stack(pot_states_list2, axis=0)
pot_start_rotvec = np.stack(pot_start_list, axis=0)
image_latents_rotvec0 = np.stack(image_latents_list0, axis=0)
image_latents_rotvec1 = np.stack(image_latents_list1, axis=0)

Found 228 rollout files in ../rollouts/robust
(661, 14)
[1/228] rollout_clean_seed0_mode2.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(572, 14)
[2/228] rollout_clean_seed0_mode3.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(567, 14)
[3/228] rollout_clean_seed100_mode2.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(583, 14)
[4/228] rollout_clean_seed100_mode3.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(583, 14)
[5/228] rollout_clean_seed10_mode2.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(687, 14)
[6/228] rollout_clean_seed10_mode3.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(593, 14)
[7/228] rollout_clean_seed110_mode2.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(616, 14)
[8/228] rollout_clean_seed110_mode3.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(694, 14)
[9/228] rollout_clean_seed120_mode2.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(563, 14)
[10/228] rollout_clean_seed120_mode3.pkl
ight blud
(700, 14)
(700, 128)
(700, 14)
(619, 14)
[11/228] rollout_clean_seed130_mode2.pk

ValueError: negative dimensions are not allowed

In [5]:
print(np.shape(expert_states_rotvec))
print(np.shape(expert_actions_rotvec))
print(np.shape(pot_states_rotvec1))
print(np.shape(pot_states_rotvec2))
print(np.shape(pot_start_rotvec))
print(np.shape(image_latents_rotvec0))
print(np.shape(image_latents_rotvec1))

(92, 700, 14)
(92, 700, 14)
(92, 700, 3)
(92, 700, 3)
(92, 6)
(92, 700, 128)
(92, 700, 128)


In [ ]:
D, T, C, H, W = image_latents_rotvec0.shape
image_latents_rotvec0 = image_latents_rotvec0.reshape(D, T, C*H*W)

D, T, C, H, W = image_latents_rotvec1.shape
image_latents_rotvec1 = image_latents_rotvec1.reshape(D, T, C*H*W)

In [6]:
print(np.shape(expert_states_rotvec))
print(np.shape(expert_actions_rotvec))
print(np.shape(pot_states_rotvec1))
print(np.shape(pot_states_rotvec2))
print(np.shape(pot_start_rotvec))
print(np.shape(image_latents_rotvec0))
print(np.shape(image_latents_rotvec1))

(92, 700, 14)
(92, 700, 14)
(92, 700, 3)
(92, 700, 3)
(92, 6)
(92, 700, 128)
(92, 700, 128)


In [7]:
import os
save_dir = "data/models/VAE_models_ICON/TrainingDataDiffusion"
os.makedirs(save_dir, exist_ok=True)
np.save(os.path.join(save_dir, "expert_states_robust.npy"), expert_states_rotvec)
np.save(os.path.join(save_dir, "expert_actions_robust.npy"), expert_actions_rotvec)
np.save(os.path.join(save_dir, "pot_states1_robust.npy"), pot_states_rotvec1)
np.save(os.path.join(save_dir, "pot_states2_robust.npy"), pot_states_rotvec2)
np.save(os.path.join(save_dir, "pot_start_robust.npy"), pot_start_rotvec)
np.save(os.path.join(save_dir, "arm1_images_latents_robust.npy"), image_latents_rotvec0)
np.save(os.path.join(save_dir, "arm2_images_latents_robust.npy"), image_latents_rotvec1)
print(f"Saved all robust data ({len(robust_files)} rollouts) to {save_dir}")


In [11]:
import os
os.path.getsize("data/models/VAE_models_ICON/arm1_images_latents.npy")

17203328

In [ ]:
# with open("rollouts/rollout_seed0_mode2.pkl", "rb") as f:
#     rollout = pkl.load(f)
#     obs = rollout["observations"]
#     actions = np.array(rollout["actions"])

#     pos0 = actions[:,:3]
#     rotvec0 = actions[:,3:6]
#     gripper0 = actions[:,6]
#     pos1 = actions[:,7:10]
#     rotvec1 = actions[:,10:13]
#     gripper1 = actions[:,13]

#     rot6d_list0 = []
#     for rv in rotvec0:
#         rot6d_list0.append(rotvec_to_rot6d(rv))
#     rot6d0 = np.array(rot6d_list0)

#     rot6d_list1 = []
#     for rv in rotvec1:
#         rot6d_list1.append(rotvec_to_rot6d(rv))
#     rot6d1 = np.array(rot6d_list1)

#     actions = np.concatenate((pos0, rot6d0, gripper0.reshape(-1, 1), pos1, rot6d1, gripper1.reshape(-1, 1)), axis=1)

#     robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
#     robot0_eef_quat = np.array([o["robot0_eef_quat"] for o in obs])
#     robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
#     robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
#     robot1_eef_quat = np.array([o["robot1_eef_quat"] for o in obs])
#     robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

#     repeats_needed = 250 - actions.shape[0]

#     repeated_last = np.tile(actions[-1], (repeats_needed, 1))
#     actions = np.vstack([actions, repeated_last])

#     repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
#     robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
#     state = robot0_eef_pos

#     repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
#     robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
#     eef_rot6d0 = []
#     for q in robot0_eef_quat:
#         eef_rot6d0.append(quat_to_rot6d(q))
#     robot0_eef_rot6d = np.array(eef_rot6d0)
#     state = np.hstack([state, robot0_eef_rot6d])


#     repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
#     robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
#     robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
#     state = np.hstack([state, robot0_gripper_pos])

#     repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
#     robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
#     state = np.hstack([state, robot1_eef_pos])

#     repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
#     robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
#     eef_rot6d1 = []
#     for q in robot1_eef_quat:
#         eef_rot6d1.append(quat_to_rot6d(q))
#     robot1_eef_rot6d = np.array(eef_rot6d1)
#     state = np.hstack([state, robot1_eef_rot6d])

#     repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
#     robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
#     robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
#     state = np.hstack([state, robot1_gripper_pos])

# print(np.shape(state))
# print(np.shape(actions))

In [14]:
expert_states_list = []
expert_actions_list = []
pot_states_list = []
for i in [2, 3]:
    for j in [0, 10, 20, 30, 40]:
        filepath = f"/content/drive/MyDrive/VAE_models_ICON/TrainingData/rollout_seed{j}_mode{i}.h5"

        with h5py.File(filepath, "r") as f:
            # obs = f["observations"]
            actions = np.array(f["actions"])
            pot = np.array(f["pot_pos"])

            pot_states_list.append(np.concatenate((pot[0], pot[1])))

            pos0 = actions[:,:3]
            rotvec0 = actions[:,3:6]
            gripper0 = actions[:,6]
            pos1 = actions[:,7:10]
            rotvec1 = actions[:,10:13]
            gripper1 = actions[:,13]

            rot6d_list0 = []
            for rv in rotvec0:
                rot6d_list0.append(rotvec_to_rot6d(rv))
            rot6d0 = np.array(rot6d_list0)

            rot6d_list1 = []
            for rv in rotvec1:
                rot6d_list1.append(rotvec_to_rot6d(rv))
            rot6d1 = np.array(rot6d_list1)

            actions = np.concatenate((pos0, rot6d0, gripper0.reshape(-1, 1), pos1, rot6d1, gripper1.reshape(-1, 1)), axis=1)

            robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
            robot0_eef_quat = np.array([o["robot0_eef_quat_site"] for o in obs])
            robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
            robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
            robot1_eef_quat = np.array([o["robot1_eef_quat_site"] for o in obs])
            robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

            repeats_needed = 400 - actions.shape[0]

            repeated_last = np.tile(actions[-1], (repeats_needed, 1))
            actions = np.vstack([actions, repeated_last])

            repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
            robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
            state = robot0_eef_pos

            repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
            robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
            eef_rot6d0 = []
            for q in robot0_eef_quat:
                eef_rot6d0.append(quat_to_rot6d(q))
            robot0_eef_rot6d = np.array(eef_rot6d0)
            state = np.hstack([state, robot0_eef_rot6d])


            repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
            robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
            robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
            state = np.hstack([state, robot0_gripper_pos])

            repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
            robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
            state = np.hstack([state, robot1_eef_pos])

            repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
            robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
            eef_rot6d1 = []
            for q in robot1_eef_quat:
                eef_rot6d1.append(quat_to_rot6d(q))
            robot1_eef_rot6d = np.array(eef_rot6d1)
            state = np.hstack([state, robot1_eef_rot6d])

            repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
            robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
            robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
            state = np.hstack([state, robot1_gripper_pos])

            expert_states_list.append(state)
            expert_actions_list.append(actions)

expert_states_rot6d = np.stack(expert_states_list, axis=0)
expert_actions_rot6d = np.stack(expert_actions_list, axis=0)
pot_states_rot6d = np.stack(pot_states_list, axis=0)

NameError: name 'h5py' is not defined

In [ ]:
print(np.shape(expert_states_rot6d))
print(np.shape(expert_actions_rot6d))
print(np.shape(pot_states_rot6d))

(20, 400, 20)
(20, 400, 20)
(20, 6)


In [ ]:
np.save("data/expert_states_rot6d_site_grippause_20.npy", expert_states_rot6d)
np.save("data/expert_actions_rot6d_site_grippause_20.npy", expert_actions_rot6d)
np.save("data/pot_states_rot6d_site_grippause_20.npy", pot_states_rot6d)